# SMART-MUSTAHIK: Sistem Pendukung Keputusan Penyaluran Zakat Presisi Menggunakan Analisis Machine Learning Berbasis QS. At-Taubah: 60

> **Karya Tulis Ilmiah Qur'an (KTIQ) SERI-FEST 2026 IPB University**  
> *Disusun oleh: Izam Rosiawan & Clairine Anargya Athallah (Telkom University Surabaya)*

---


In [ ]:
# Import pustaka manipulasi data & pemodelan
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'


In [ ]:
# Memuat & Membersihkan Dataset BPS Jawa Timur (2020-2024)
csv_path = os.path.join('data', 'dataset_jatim_2020_2024.csv')
df_raw = pd.read_csv(csv_path)

num_cols = [
    'total_population', 'number_of_poor_people', 'poverty_percentage',
    'school_participation_rate', 'sanitation_access_percent',
    'drinking_water_access_percent', 'proper_housing_percent',
    'unemployment_rate', 'gdp_per_capita', 'population_density', 'hdi'
]

df = df_raw.copy()
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Dataset BPS Jawa Timur Dimuat: {len(df)} Observasi (38 Kab/Kota x 5 Tahun)")
df.head()


In [ ]:
# Formulasi Ground Truth Mustahik (QS. At-Taubah: 60) & Had Kifayah
HAD_KIFAYAH = 1950000  # Had Kifayah bulanan per kapita (Rp)

df['pendapatan_per_kapita_bulan'] = (df['gdp_per_capita'] * 1000) / 12.0

# Status Prioritas Mustahik (y=1) vs Non-Mustahik (y=0)
df['status_mustahik'] = np.where(
    (df['poverty_percentage'] > 9.5) | (df['pendapatan_per_kapita_bulan'] < HAD_KIFAYAH),
    1, 0
)

dist = df['status_mustahik'].value_counts()
print(f"Mustahik (Fakir/Miskin) : {dist.get(1, 0)} Observasi ({dist.get(1, 0)/len(df)*100:.1f}%)")
print(f"Non-Mustahik (Mampu)    : {dist.get(0, 0)} Observasi ({dist.get(0, 0)/len(df)*100:.1f}%)")


In [ ]:
# Pemilihan Fitur Prediktor & Pemisahan Dataset Stratified Split (75:25)
# 6 Fitur Prediktor (Eksklusi variabel rawan kebocoran target: number_of_poor_people & total_population)
features = [
    'hdi', 'sanitation_access_percent', 'drinking_water_access_percent',
    'school_participation_rate', 'unemployment_rate', 'population_density'
]

X = df[features]
y = df['status_mustahik']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Imputasi nilai kosong berbasis Median (Fit murni pada sampel latih)
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=features, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=features, index=X_test.index)

# Model Usulan: Random Forest Classifier
model_smart = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=SEED)
model_smart.fit(X_train_imp, y_train)
y_pred_smart = model_smart.predict(X_test_imp)

# Baseline Simulasi Random Label Noise (16.67% noise = 8 sampel acak dari 48 sampel uji)
y_pred_noise = y_test.copy().values
n_noise = 8
noise_idx = np.random.choice(len(y_pred_noise), size=n_noise, replace=False)
y_pred_noise[noise_idx] = 1 - y_pred_noise[noise_idx]


In [ ]:
# Evaluasi Performa & Kalkulasi Inclusion/Exclusion Error Zakat
def eval_performance(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred) * 100
    rec = recall_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred) * 100
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    inc_err = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    exc_err = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0.0
    return {
        'Metode': name,
        'Akurasi (%)': round(acc, 2),
        'Precision (%)': round(prec, 2),
        'Recall (%)': round(rec, 2),
        'F1-Score (%)': round(f1, 2),
        'Inclusion Error Rate (%)': round(inc_err, 2),
        'Exclusion Error Rate (%)': round(exc_err, 2)
    }

res_noise = eval_performance(y_test, y_pred_noise, 'Baseline Noise')
res_smart = eval_performance(y_test, y_pred_smart, 'SMART-MUSTAHIK')

df_eval = pd.DataFrame([res_noise, res_smart]).set_index('Metode')
df_eval.to_csv(os.path.join('data', 'evaluasi_performa.csv'))
df_eval


In [ ]:
# Visualisasi Publikasi Grafik 300 DPI
# 1. Plot Utama Perbandingan Performa & Error Zakat
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
colors = ['#d9534f', '#2b8a3e']

df_p1 = df_eval[['Akurasi (%)', 'F1-Score (%)']].reset_index().melt(id_vars='Metode', var_name='Metrik', value_name='Persentase (%)')
sns.barplot(data=df_p1, x='Metrik', y='Persentase (%)', hue='Metode', ax=axes[0], palette=colors)
axes[0].set_title('Perbandingan Akurasi & F1-Score Klasifikasi', fontsize=12, fontweight='bold', pad=10)
axes[0].set_ylim(0, 115)
axes[0].set_xlabel('')
axes[0].set_ylabel('Persentase (%)', fontsize=11)
axes[0].legend(title='', loc='upper left', frameon=True)
for p in axes[0].patches:
    h = p.get_height()
    if h > 0:
        axes[0].annotate(f'{h:.2f}%', (p.get_x() + p.get_width() / 2., h + 1.5), ha='center', va='bottom', fontsize=9.5, fontweight='bold')

df_p2 = df_eval[['Inclusion Error Rate (%)', 'Exclusion Error Rate (%)']].reset_index().melt(id_vars='Metode', var_name='Tipe Error', value_name='Persentase Error (%)')
sns.barplot(data=df_p2, x='Tipe Error', y='Persentase Error (%)', hue='Metode', ax=axes[1], palette=colors)
axes[1].set_title('Perbandingan Inclusion & Exclusion Error Zakat', fontsize=12, fontweight='bold', pad=10)
axes[1].set_ylim(0, 25)
axes[1].set_xlabel('')
axes[1].set_ylabel('Persentase Error (%)', fontsize=11)
axes[1].legend(title='', loc='upper right', frameon=True)
for p in axes[1].patches:
    h = p.get_height()
    if h > 0:
        axes[1].annotate(f'{h:.2f}%', (p.get_x() + p.get_width() / 2., h + 0.6), ha='center', va='bottom', fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join('images', 'grafik_hasil_smart_mustahik.png'), dpi=300, bbox_inches='tight')
plt.show()

# 2. Plot Feature Importance
importances = model_smart.feature_importances_
feat_dict = {
    'hdi': 'Indeks Pembangunan Manusia (IPM)',
    'sanitation_access_percent': 'Akses Sanitasi Layak (%)',
    'drinking_water_access_percent': 'Akses Air Minum Layak (%)',
    'school_participation_rate': 'Angka Partisipasi Sekolah (%)',
    'unemployment_rate': 'Tingkat Pengangguran Terbuka (%)',
    'population_density': 'Kepadatan Penduduk (Jiwa/km²)'
}
feat_names = [feat_dict[f] for f in features]
df_imp = pd.DataFrame({'Fitur': feat_names, 'Importance': importances}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(df_imp['Fitur'], df_imp['Importance'], color='#2b8a3e', edgecolor='#1b5e20', alpha=0.85)
plt.title('Tingkat Kepentingan Fitur (Feature Importance) - SMART-MUSTAHIK', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Nilai Relative Importance', fontsize=11)
for i, v in enumerate(df_imp['Importance']):
    plt.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9.5, fontweight='bold')
plt.xlim(0, max(importances) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join('images', 'feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()

# 3. Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred_smart)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Non-Mustahik (0)', 'Prioritas Mustahik (1)'],
            yticklabels=['Non-Mustahik (0)', 'Prioritas Mustahik (1)'],
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Matriks Konfusi Model SMART-MUSTAHIK', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Prediksi Model', fontsize=11, fontweight='bold')
plt.ylabel('Ground Truth (BPS & Had Kifayah)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('images', 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
